In [ ]:
import csv
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Part 01

1. Read data

In [5]:
PROJECT_ROOT = Path("..")
DATA_RAW = PROJECT_ROOT / "data" / "raw"
data_path = DATA_RAW / "nasdaq_exteral_data.csv" 
sample_path = DATA_RAW / "nasdaq_sample_rows_5000.csv"  

# read data row by row
max_rows = 5000
count = 0
with open(data_path, "r", newline="", encoding="utf-8", errors="ignore") as f_in, \
     open(sample_path, "w", newline="", encoding="utf-8") as f_out:
    
    reader = csv.reader(f_in)
    writer = csv.writer(f_out)
    
    for row in reader:
        writer.writerow(row)
        count += 1
        if count >= max_rows:
            break

# verify
print("Writtern records:", count)

# preview news
news_preview = pd.read_csv(sample_path, nrows=5)
print(news_preview.columns)

news = pd.read_csv(sample_path)
news["Stock_symbol"].value_counts().head(10)

Writtern records: 5000
Index(['Unnamed: 0', 'Date', 'Article_title', 'Stock_symbol', 'Url',
       'Publisher', 'Author', 'Article', 'Lsa_summary', 'Luhn_summary',
       'Textrank_summary', 'Lexrank_summary'],
      dtype='object')


Stock_symbol
AAL     2962
AA      1525
A        379
AADR      77
AACG      49
AAAU       7
Name: count, dtype: int64

2. Construct event list

In [6]:
TARGET_TICKER = "AAL"
news["Date"] = pd.to_datetime(news["Date"])
news["event_date"] = news["Date"].dt.date 

news_t = news[news["Stock_symbol"] == TARGET_TICKER].copy()

# the lists of date where our target ticker occurs on news
events_small = (
    news_t
    .drop_duplicates(subset=["Stock_symbol", "event_date"])
    .loc[:, ["Stock_symbol", "event_date"]]
    .rename(columns={"Stock_symbol": "ticker"})
    .sort_values(["ticker", "event_date"])
    .reset_index(drop=True)
)

events_small.head(), len(events_small)

(  ticker  event_date
 0    AAL  2020-11-23
 1    AAL  2020-11-24
 2    AAL  2020-11-25
 3    AAL  2020-11-26
 4    AAL  2020-11-27,
 809)

3. Stock prices

In [7]:
PRICE_DIR = DATA_RAW / "full_history"
price_path = PRICE_DIR / "AAL.csv"

prices_small = pd.read_csv(price_path)
print(prices_small.columns)

prices_small["date"] = pd.to_datetime(prices_small["date"]).dt.date
prices_small = prices_small.sort_values("date").reset_index(drop=True)
prices_small.head(), prices_small.tail()

Index(['date', 'open', 'high', 'low', 'close', 'adj close', 'volume'], dtype='object')


(         date       open       high        low      close  adj close   volume
 0  2005-09-27  21.049999  21.400000  19.100000  19.299999  18.194910   961200
 1  2005-09-28  19.299999  20.530001  19.200001  20.500000  19.326199  5747900
 2  2005-09-29  20.400000  20.580000  20.100000  20.209999  19.052801  1078200
 3  2005-09-30  20.260000  21.049999  20.180000  21.010000  19.806999  3123300
 4  2005-10-03  20.900000  21.750000  20.900000  21.500000  20.268938  1057900,
             date   open   high    low  close  adj close    volume
 4590  2023-12-21  14.21  14.43  14.20  14.35      14.35  30372600
 4591  2023-12-22  14.38  14.40  14.21  14.31      14.31  25169900
 4592  2023-12-26  14.25  14.26  14.04  14.11      14.11  22157900
 4593  2023-12-27  14.10  14.18  13.91  13.99      13.99  23428500
 4594  2023-12-28  13.92  14.04  13.82  13.98      13.98  17069900)

4. Calculate returns

In [8]:
prices_small["ret"] = prices_small["close"].pct_change()
prices_small["trading_idx"] = np.arange(len(prices_small))
idx_by_date = dict(zip(prices_small["date"], prices_small["trading_idx"]))
prices_small.head()

,date,open,high,low,close,adj close,volume,ret,trading_idx
0,2005-09-27,21.049999,21.400000,19.100000,19.299999,18.194910,961200,NaN,0
1,2005-09-28,19.299999,20.530001,19.200001,20.500000,19.326199,5747900,0.062176,1
2,2005-09-29,20.400000,20.580000,20.100000,20.209999,19.052801,1078200,-0.014146,2
3,2005-09-30,20.260000,21.049999,20.180000,21.010000,19.806999,3123300,0.039584,3
4,2005-10-03,20.900000,21.750000,20.900000,21.500000,20.268938,1057900,0.023322,4


5. Calculate CAR

In [51]:
def car_m1p3(event_date, prices_df, idx_map):
    if event_date not in idx_map:
        return np.nan
    
    center = idx_map[event_date]
    start_idx = center - 1
    end_idx = center + 3

    mask = (prices_df["trading_idx"] >= start_idx) & (prices_df["trading_idx"] <= end_idx)
    window = prices_df[mask]

    return window["ret"].sum()

def add_car_to_events(events_df, prices_df, idx_map):
    cars = []
    for d in events_df["event_date"]:
        car = car_m1p3(d, prices_df, idx_map)
        cars.append(car)
    out = events_df.copy()
    out["car_m1p3"] = cars
    return out

events_with_car = add_car_to_events(events_small, prices_small, idx_by_date)
events_with_car.head()

,ticker,event_date,car_m1p3
0,AAL,2020-11-23,0.165569
1,AAL,2020-11-24,0.129155
2,AAL,2020-11-25,0.056861
3,AAL,2020-11-26,NaN
4,AAL,2020-11-27,0.005286


# Part 02:

1. Add article to event list

In [10]:
# events & news from part 1
events = events_with_car.copy()
events["event_date"] = pd.to_datetime(events["event_date"]).dt.date

print(events.columns)
print(news.columns)

# keep only news of AAL
news_aal = news[news["Stock_symbol"] == "AAL"].copy()
news_aal_unique = (
    news_aal
    .sort_values("Date")
    .drop_duplicates(subset=["Stock_symbol", "event_date"])
)
news_aal_unique = news_aal_unique.rename(columns={"Stock_symbol": "ticker"})

# merge (ticker, event_Date, article_title)
events_merged = events.merge(
    news_aal_unique[["ticker", "event_date", "Article_title"]],
    on=["ticker", "event_date"],
    how="left"
)
events_merged[["ticker", "event_date", "car_m1p3", "Article_title"]].head()

Index(['ticker', 'event_date', 'car_m1p3'], dtype='object')
Index(['Unnamed: 0', 'Date', 'Article_title', 'Stock_symbol', 'Url',
       'Publisher', 'Author', 'Article', 'Lsa_summary', 'Luhn_summary',
       'Textrank_summary', 'Lexrank_summary', 'event_date'],
      dtype='object')


,ticker,event_date,car_m1p3,Article_title
0,AAL,2020-11-23,0.165569,"Pre-Market Most Active for Nov 23, 2020 : FCEL..."
1,AAL,2020-11-24,0.129155,"Colombia coal miner Cerrejon, union to hold te..."
2,AAL,2020-11-25,0.056861,EXCLUSIVE-White House considers lifting Europe...
3,AAL,2020-11-26,NaN,3 Cheap Stocks to Buy Before the Economy Makes...
4,AAL,2020-11-27,0.005286,Looking For Cheap Stocks To Buy On Black Frida...


2. Create label

In [44]:
import numpy as np
import pandas as pd

# 1. 先保证 event_date 是日期类型
events_merged["event_date"] = pd.to_datetime(events_merged["event_date"]).dt.date

# 2. 去掉 CAR 为 NaN 的行
events_clf = events_merged.dropna(subset=["car_m1p3"]).copy()

# 3. 设一个阈值，过滤掉绝对值太小的 CAR（噪音区）
THRESH = 0.01  # 你可以试 0.005、0.02 等

mask_keep = (events_clf["car_m1p3"] >= THRESH) | (events_clf["car_m1p3"] <= -THRESH)
events_clf = events_clf[mask_keep].copy()

# 4. 重新构造标签：CAR>0 为 1，CAR<0 为 0
events_clf["label_up"] = (events_clf["car_m1p3"] > 0).astype(int)

print(events_clf["label_up"].value_counts())
print(events_clf["label_up"].value_counts(normalize=True))  # 看一下正负比例
events_clf[["event_date", "car_m1p3", "label_up", "Article_title"]].head()


label_up
0    324
1    317
Name: count, dtype: int64
label_up
0    0.50546
1    0.49454
Name: proportion, dtype: float64


,event_date,car_m1p3,label_up,Article_title
0,2020-11-23,0.165569,1,"Pre-Market Most Active for Nov 23, 2020 : FCEL..."
1,2020-11-24,0.129155,1,"Colombia coal miner Cerrejon, union to hold te..."
2,2020-11-25,0.056861,1,EXCLUSIVE-White House considers lifting Europe...
7,2020-11-30,0.079961,1,Why Airline Shares Are Falling Today
8,2020-12-01,0.096550,1,Airlines would receive $17 bln in new COVID-19...


3. Extract text label

In [45]:
import sys
print(sys.executable)
!"{sys.executable}" -m pip install transformers

/opt/anaconda3/bin/python


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [46]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

# choose device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)

# FinBERT
model_name = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

device = cpu


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [47]:
def finbert_sentiment_features(texts, batch_size=16, max_length=128):
    all_probs = []
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits              # (batch, 3)
            probs = F.softmax(logits, dim=-1)    # turn into probability

        all_probs.append(probs.cpu().numpy())

    return np.vstack(all_probs)

In [48]:
events_clf["Article_title"] = events_clf["Article_title"].fillna("").astype(str)

texts = events_clf["Article_title"].tolist()
labels = events_clf["label_up"].values

X_finbert = finbert_sentiment_features(texts, batch_size=16, max_length=64)
X_finbert.shape  # (N, 3)

(641, 3)

4. Train model

In [52]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# 把 event_date 转成 pandas Timestamp，方便比较
events_clf["event_date_ts"] = pd.to_datetime(events_clf["event_date"])

# 设一个时间点：之前的事件做 train，之后的事件做 test
cut_date = pd.Timestamp("2023-01-01")  

train_mask = events_clf["event_date_ts"] < cut_date
test_mask = ~train_mask

X_train = X_finbert[train_mask.values]
X_test  = X_finbert[test_mask.values]

y_train = events_clf.loc[train_mask, "label_up"].values
y_test  = events_clf.loc[test_mask, "label_up"].values

print("train size:", X_train.shape[0], "test size:", X_test.shape[0])

clf = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",  # 处理类别不平衡
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print("ACC:", accuracy_score(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_proba))
print("\nReport:\n", classification_report(y_test, y_pred))


train size: 446 test size: 195
ACC: 0.558974358974359
AUC: 0.5531869987336429

Report:
               precision    recall  f1-score   support

           0       0.60      0.51      0.55       103
           1       0.53      0.61      0.57        92

    accuracy                           0.56       195
   macro avg       0.56      0.56      0.56       195
weighted avg       0.56      0.56      0.56       195

